# MOHIM motif dataset → ACE-Step LoRA

이 노트북은 별도 motif extraction 노트북에서 생성한 `motif_dataset`으로 manifest를 만들고 ACE-Step LoRA를 학습합니다.

## 0. Drive 마운트와 저장소 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'mean-centering'
REPO_DIR = Path('/content/MOHIM')

if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt

# 첫 설치 후 import 오류가 나면 런타임을 한 번 재시작하고 이 셀부터 다시 실행하세요.

## 1. LoRA 경로와 실행 설정

In [ ]:
from pathlib import Path
import json

LORA_VERSION = 'v6_repeated_motif'  # start/end anchored repeated-motif adapter
SMOKE_VERSION = f'{LORA_VERSION}_smoke'
LORA_RUN_ROOT = Path('/content/drive/MyDrive/MOHIM/lora_run')
LORA_OUTPUT_DIR = LORA_RUN_ROOT / LORA_VERSION
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/MOHIM/mohim_lora_checkpoints')
DRIVE_CHECKPOINT_DIR = DRIVE_CHECKPOINT_ROOT / LORA_VERSION
INFERENCE_OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/inference') / LORA_VERSION

OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MOTIF_VERSION_PATH = OUTPUT_DIR / '.mohim_motif_version'
MANIFEST_PATH = OUTPUT_DIR / 'dual_stream_manifest.json'
TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/dual_stream_tensors_mean_centered')
TENSOR_SCHEMA = 'per_stem_motif_v6_repeated'
LEGACY_TENSOR_SCHEMA = 'per_stem_motif_v5'
TENSOR_SCHEMA_PATH = TENSOR_DIR / '.mohim_schema'
SMOKE_SONGS = 3
MAX_TRACK_DURATION = 300.0
RESET_TENSORS = False
RESET_LORA_RUN = False
DEVICE = 'cuda'

assert OUTPUT_DIR.is_dir(), f'motif dataset이 없습니다: {OUTPUT_DIR}'
assert MOTIF_VERSION_PATH.is_file(), (
    '별도 모티프 추출 노트북으로 생성한 데이터셋이 아닙니다: '
    f'{MOTIF_VERSION_PATH}'
)
motif_dataset_config = json.loads(MOTIF_VERSION_PATH.read_text(encoding='utf-8'))
allowed_track_ids = motif_dataset_config.get('accepted_track_ids')
assert isinstance(allowed_track_ids, list) and allowed_track_ids, (
    'motif dataset에 학습 가능한 track 목록이 없습니다.'
)
motif_dataset_version = json.dumps(
    motif_dataset_config, ensure_ascii=False, sort_keys=True, separators=(',', ':')
)
tensor_cache_version = json.dumps(
    {
        'tensor_schema': TENSOR_SCHEMA,
        'motif_dataset_version': motif_dataset_version,
    },
    ensure_ascii=False,
    sort_keys=True,
)
tensor_cache_ready = (
    not RESET_TENSORS
    and TENSOR_SCHEMA_PATH.is_file()
    and TENSOR_SCHEMA_PATH.read_text(encoding='utf-8').strip() == tensor_cache_version
    and next(TENSOR_DIR.glob('*.pt'), None) is not None
)
legacy_tensor_cache_ready = False
if not RESET_TENSORS and TENSOR_SCHEMA_PATH.is_file() and next(TENSOR_DIR.glob('*.pt'), None) is not None:
    try:
        previous_tensor_cache = json.loads(TENSOR_SCHEMA_PATH.read_text(encoding='utf-8'))
        legacy_tensor_cache_ready = (
            previous_tensor_cache.get('tensor_schema') == LEGACY_TENSOR_SCHEMA
            and previous_tensor_cache.get('motif_dataset_version') == motif_dataset_version
        )
    except (json.JSONDecodeError, OSError):
        legacy_tensor_cache_ready = False
print('motif dataset:', OUTPUT_DIR)
print('motif dataset version:', motif_dataset_version)
print('existing tensor cache:', tensor_cache_ready)
print('motif-only migration available:', legacy_tensor_cache_ready)


## 2. ACE-Step dual-stream manifest 생성

In [ ]:
import json
from mohim.manifest import build_dual_stream_manifest

if tensor_cache_ready and MANIFEST_PATH.is_file():
    print('기존 dual-stream tensor와 manifest를 사용합니다.')
else:
    manifest = build_dual_stream_manifest(
        OUTPUT_DIR,
        MANIFEST_PATH,
        allowed_track_ids=allowed_track_ids,
    )
    print('usable samples:', manifest['metadata']['num_samples'])
    print('manifest:', MANIFEST_PATH)
    if manifest['samples']:
        print(json.dumps(manifest['samples'][0], ensure_ascii=False, indent=2)[:3000])

## 3. 공식 ACE-Step clone 및 dual-stream 패치 적용

In [ ]:
from mohim.trainer import (
    DEFAULT_REVISION,
    apply_acestep_patch,
    ensure_acestep_repo,
    install_acestep,
)

ACESTEP_DIR = Path('/content/ACE-Step-1.5')
PATCH_FILE = REPO_DIR / 'patches/ace-step-1.5-dual-stream.patch'
ACESTEP_DIR = ensure_acestep_repo(ACESTEP_DIR, revision=DEFAULT_REVISION)
apply_acestep_patch(ACESTEP_DIR, PATCH_FILE)
INSTALL_ACESTEP = True
if INSTALL_ACESTEP:
    install_acestep(ACESTEP_DIR)
# Colab에서 검증된 PyTorch 2.10 조합과 기본 패키지 버전을 복구합니다.
%pip uninstall -q -y huggingface-hub
%pip install -q --force-reinstall --no-cache-dir --no-deps \
    huggingface-hub==0.36.0 requests==2.32.4 fsspec==2025.3.0 \
    torchcodec==0.10.0 torchao==0.16.0

# 재설치 전 실패한 Hugging Face import가 현재 커널에 남아 있으면 제거합니다.
import importlib
import sys
for module_name in list(sys.modules):
    if module_name == 'huggingface_hub' or module_name.startswith('huggingface_hub.'):
        del sys.modules[module_name]
    elif module_name == 'transformers' or module_name.startswith('transformers.'):
        del sys.modules[module_name]
    elif module_name == 'peft' or module_name.startswith('peft.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
print('ACE-Step directory:', ACESTEP_DIR)

## 4. Dual-stream tensor 전처리

In [ ]:
import shutil
import subprocess
import sys
import time

CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
PREPROCESS_RUN_DIR = ACESTEP_DIR / 'mohim_preprocess_run'
MODEL_VARIANT = 'base'
MAX_DURATION = MAX_TRACK_DURATION

if tensor_cache_ready:
    print('기존 dual-stream tensor를 사용해 전처리를 건너뜁니다:', TENSOR_DIR)
elif legacy_tensor_cache_ready:
    import torch
    from acestep.training.dataset_builder_modules.preprocess_audio import load_audio_stereo
    from acestep.training_v2.dual_stream_preprocess import (
        MOTIF_CONDITION_SCHEMA, encode_motif_condition_latents,
    )
    from acestep.training_v2.model_loader import load_vae, unload_models

    manifest_data = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    samples_by_audio = {
        str(Path(sample['audio_path']).resolve()): sample
        for sample in manifest_data['samples']
    }
    tensor_files = sorted(TENSOR_DIR.glob('*.pt'))
    assert tensor_files, '마이그레이션할 기존 tensor가 없습니다.'
    vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
    migration_started = time.monotonic()
    migrated = 0
    try:
        for index, tensor_path in enumerate(tensor_files, start=1):
            item = torch.load(tensor_path, map_location='cpu', weights_only=True)
            if item.get('motif_condition_schema') == MOTIF_CONDITION_SCHEMA:
                status = 'already migrated'
            else:
                metadata = item.get('metadata', {})
                audio_key = str(Path(metadata.get('audio_path', '')).resolve())
                sample = samples_by_audio.get(audio_key)
                if sample is None:
                    raise KeyError(f'manifest에서 tensor 원본을 찾지 못했습니다: {tensor_path.name}')
                target_audio, _ = load_audio_stereo(
                    sample['accompaniment_target_audio'], 48000, MAX_DURATION
                )
                motif_latents = encode_motif_condition_latents(
                    sample['motif_seed_audio'],
                    vae,
                    torch.bfloat16,
                    target_samples=target_audio.shape[-1],
                    motif_start_sec=float(sample['motif_start_sec']),
                    motif_end_sec=float(sample['motif_end_sec']),
                )
                expected_frames = item['accompaniment_target_latents'].shape[0]
                if motif_latents.shape[0] != expected_frames:
                    raise ValueError(
                        f'{tensor_path.name}: motif/target frame mismatch '
                        f'{motif_latents.shape[0]} != {expected_frames}'
                    )
                item['motif_seed_latents'] = motif_latents
                item['motif_seed_attention_mask'] = torch.ones(
                    motif_latents.shape[0], dtype=torch.bfloat16
                )
                item['motif_condition_schema'] = MOTIF_CONDITION_SCHEMA
                temp_path = tensor_path.with_name(tensor_path.name + '.repeated.tmp')
                torch.save(item, temp_path)
                temp_path.replace(tensor_path)
                migrated += 1
                status = 'motif latent replaced'
                del target_audio, motif_latents
            elapsed = time.monotonic() - migration_started
            remaining = len(tensor_files) - index
            eta_seconds = elapsed / index * remaining
            print(
                f'[{index}/{len(tensor_files)}] {status}; completed={index}, '
                f'remaining={remaining}, ETA={eta_seconds / 60:.1f}m',
                flush=True,
            )
            del item
    finally:
        unload_models(vae)
        del vae
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    TENSOR_SCHEMA_PATH.write_text(tensor_cache_version + '\n', encoding='utf-8')
    tensor_cache_ready = True
    print(f'motif-only migration complete: {migrated}/{len(tensor_files)} tensors updated')
else:
    # 알 수 없는 schema의 기존 tensor는 자동 삭제하지 않습니다.
    tensor_cache_changed = (
        TENSOR_DIR.exists()
        and (
            not TENSOR_SCHEMA_PATH.is_file()
            or TENSOR_SCHEMA_PATH.read_text(encoding='utf-8').strip() != tensor_cache_version
        )
    )
    existing_tensor_files = next(TENSOR_DIR.glob('*.pt'), None) is not None
    if tensor_cache_changed and existing_tensor_files and not RESET_TENSORS:
        raise RuntimeError(
            '기존 tensor schema를 안전하게 마이그레이션할 수 없습니다. '
            '삭제 후 전체 재생성이 필요할 때만 RESET_TENSORS=True로 실행하세요.'
        )
    if (RESET_TENSORS or tensor_cache_changed) and TENSOR_DIR.exists():
        shutil.rmtree(TENSOR_DIR)
    TENSOR_DIR.mkdir(parents=True, exist_ok=True)
    PREPROCESS_RUN_DIR.mkdir(parents=True, exist_ok=True)

    preprocess_result = subprocess.run(
        [
            sys.executable, 'train.py', 'fixed',
            '--checkpoint-dir', str(CHECKPOINT_DIR),
            '--model-variant', MODEL_VARIANT,
            '--dataset-dir', str(TENSOR_DIR),
            '--output-dir', str(PREPROCESS_RUN_DIR),
            '--preprocess', '--dual-stream',
            '--dataset-json', str(MANIFEST_PATH),
            '--tensor-output', str(TENSOR_DIR),
            '--max-duration', str(MAX_DURATION),
            '--device', DEVICE, '--precision', 'bf16',
        ],
        cwd=ACESTEP_DIR,
        check=False,
    )
    if preprocess_result.returncode != 0:
        raise RuntimeError(
            f'dual-stream 전처리가 종료 코드 {preprocess_result.returncode}로 실패했습니다.'
        )
    TENSOR_SCHEMA_PATH.write_text(tensor_cache_version + '\n', encoding='utf-8')
    tensor_cache_ready = True

In [ ]:
import torch

tensor_files = sorted(TENSOR_DIR.glob('*.pt'))
assert tensor_files, '전처리 tensor가 생성되지 않았습니다.'
item = torch.load(tensor_files[0], map_location='cpu', weights_only=True)
assert item.get('motif_condition_schema') == 'anchored_repeated_v1', (
    '반복 모티프 tensor schema가 적용되지 않았습니다.'
)
required = [
    'motif_seed_latents', 'motif_seed_attention_mask',
    'accompaniment_target_latents', 'accompaniment_target_attention_mask',
    'vocal_target_latents', 'vocal_target_attention_mask',
    'accompaniment_encoder_hidden_states', 'accompaniment_encoder_attention_mask',
    'vocal_encoder_hidden_states', 'vocal_encoder_attention_mask',
]
for key in required:
    assert key in item, f'missing tensor: {key}'
    print(key, tuple(item[key].shape))

## 5. LoRA 학습

전체 tensor 중 3개로 1 epoch smoke test를 먼저 실행해 checkpoint 저장과 inference adapter 로딩을 검증합니다. 통과한 뒤 별도의 본학습 셀을 실행합니다. 곡 단위 80/20 train-validation 고정 분할을 사용합니다.

In [ ]:
import gc
import json
import os
import shlex
import shutil
import torch

LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_dual_stream_tensors'
SMOKE_LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_dual_stream_tensors_smoke'
LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{LORA_VERSION}'
SMOKE_LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{SMOKE_VERSION}'
SMOKE_LORA_OUTPUT_DIR = LORA_RUN_ROOT / SMOKE_VERSION
SMOKE_DRIVE_CHECKPOINT_DIR = DRIVE_CHECKPOINT_ROOT / SMOKE_VERSION

# 전체 tensor는 한 번만 준비하고, 그중 3개를 smoke 전용 로컬 디렉터리로 격리합니다.
for local_tensor_dir in (LOCAL_TENSOR_DIR, SMOKE_LOCAL_TENSOR_DIR):
    if local_tensor_dir.exists():
        shutil.rmtree(local_tensor_dir)
shutil.copytree(TENSOR_DIR, LOCAL_TENSOR_DIR)
all_tensor_files = sorted(LOCAL_TENSOR_DIR.glob('*.pt'))
assert len(all_tensor_files) >= SMOKE_SONGS, (
    f'smoke test에 필요한 tensor가 부족합니다: {len(all_tensor_files)}/{SMOKE_SONGS}'
)
SMOKE_LOCAL_TENSOR_DIR.mkdir(parents=True)
for tensor_path in all_tensor_files[:SMOKE_SONGS]:
    shutil.copy2(tensor_path, SMOKE_LOCAL_TENSOR_DIR / tensor_path.name)
assert len(list(SMOKE_LOCAL_TENSOR_DIR.glob('*.pt'))) == SMOKE_SONGS

# smoke는 항상 처음부터 실행하며 본학습 v3 결과에는 손대지 않습니다.
for smoke_path in (SMOKE_LOCAL_RUN_DIR, SMOKE_LORA_OUTPUT_DIR, SMOKE_DRIVE_CHECKPOINT_DIR):
    if smoke_path.exists():
        shutil.rmtree(smoke_path)
SMOKE_LOCAL_RUN_DIR.mkdir(parents=True)
SMOKE_DRIVE_CHECKPOINT_DIR.mkdir(parents=True)

if RESET_LORA_RUN:
    for main_path in (LOCAL_RUN_DIR, LORA_OUTPUT_DIR, DRIVE_CHECKPOINT_DIR):
        if main_path.exists():
            shutil.rmtree(main_path)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_epoch(path):
    try:
        state = torch.load(path / 'training_state.pt', map_location='cpu', weights_only=False)
        return int(state.get('epoch', -1))
    except Exception:
        return -1

def valid_resume_checkpoint(path):
    required = (
        'training_state.pt', 'adapter_model.safetensors',
        'dual_stream_conditioner.pt', 'run_config.json', 'rng_state.pt',
    )
    return path.is_dir() and all((path / name).is_file() for name in required) and checkpoint_epoch(path) >= 0

def prepare_main_resume():
    candidates = [
        DRIVE_CHECKPOINT_DIR / 'resume_latest',
        DRIVE_CHECKPOINT_DIR / 'resume_latest.previous',
        *DRIVE_CHECKPOINT_DIR.glob('epoch_*'),
    ]
    valid = [path for path in candidates if valid_resume_checkpoint(path)]
    latest = max(valid, key=checkpoint_epoch, default=None)
    if latest is None:
        print(f'no {LORA_VERSION} checkpoint found; starting from epoch 1', flush=True)
        return None
    local_resume_dir = LOCAL_RUN_DIR / 'resume_checkpoint'
    if local_resume_dir.exists():
        shutil.rmtree(local_resume_dir)
    shutil.copytree(latest, local_resume_dir)
    print(f'resuming {LORA_VERSION} from: {latest}', flush=True)
    return local_resume_dir

### 5.1 스모크 테스트

In [ ]:
os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(SMOKE_DRIVE_CHECKPOINT_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
%cd {ACESTEP_DIR}
print(f'SMOKE TEST: {SMOKE_SONGS} samples, 1 epoch', flush=True)
!python -u train.py --plain --yes fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{SMOKE_LOCAL_TENSOR_DIR}" \
  --output-dir "{SMOKE_LOCAL_RUN_DIR}" \
  --dual-stream --attention-type both \
  --rank 64 --alpha 128 --dropout 0.1 \
  --batch-size 1 --gradient-accumulation 1 \
  --epochs 1 --save-every 1 \
  --lr 0.0003 --val-split 0.2 \
  --shift 1.0 --num-inference-steps 50 \
  --optimizer-type adamw8bit --scheduler-type cosine_restarts \
  --warmup-steps 100 --weight-decay 0.01 \
  --max-grad-norm 1.0 --seed 42 \
  --num-workers 0 --log-every 1 \
  --device "{DEVICE}" --precision bf16
if _exit_code != 0:
    raise RuntimeError(f'smoke 학습이 종료 코드 {_exit_code}로 실패했습니다.')

from peft import PeftModel
from acestep.training_v2.dual_stream import attach_dual_stream_conditioner
from acestep.training_v2.model_loader import load_decoder_for_training

smoke_checkpoint = SMOKE_LOCAL_RUN_DIR / 'resume_latest'
smoke_final_dir = SMOKE_LOCAL_RUN_DIR / 'final'
smoke_required = (
    'training_state.pt', 'adapter_model.safetensors', 'adapter_config.json',
    'dual_stream_config.json', 'dual_stream_conditioner.pt',
    'run_config.json', 'rng_state.pt',
)
smoke_missing = [name for name in smoke_required if not (smoke_checkpoint / name).is_file()]
assert not smoke_missing, f'smoke checkpoint 파일이 없습니다: {smoke_missing}'

smoke_model = load_decoder_for_training(
    CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16'
)
smoke_model.decoder = PeftModel.from_pretrained(
    smoke_model.decoder, str(smoke_final_dir), is_trainable=False
)
smoke_conditioner_config = json.loads(
    (smoke_final_dir / 'dual_stream_config.json').read_text(encoding='utf-8')
)
assert smoke_conditioner_config.pop('schema_version') == 2, '구형 dual-stream checkpoint입니다.'
smoke_conditioner = attach_dual_stream_conditioner(
    smoke_model.decoder, **smoke_conditioner_config
)
smoke_conditioner.load_state_dict(torch.load(
    smoke_final_dir / 'dual_stream_conditioner.pt', map_location='cpu', weights_only=True,
))
del smoke_conditioner, smoke_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
shutil.copytree(SMOKE_LOCAL_RUN_DIR, SMOKE_LORA_OUTPUT_DIR, dirs_exist_ok=True)
print('SMOKE TEST PASSED', flush=True)

### 5.2 본학습

In [ ]:
main_resume_dir = prepare_main_resume()
main_resume_option = (
    f'--resume-from {shlex.quote(str(main_resume_dir))}' if main_resume_dir else ''
)
os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(DRIVE_CHECKPOINT_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
%cd {ACESTEP_DIR}
!python -u train.py --plain --yes fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{LOCAL_TENSOR_DIR}" \
  --output-dir "{LOCAL_RUN_DIR}" \
  --dual-stream --attention-type both \
  --rank 64 --alpha 128 --dropout 0.1 \
  --batch-size 1 --gradient-accumulation 1 \
  --epochs 100 --save-every 10 \
  --lr 0.0003 --val-split 0.2 \
  --shift 1.0 --num-inference-steps 50 \
  --optimizer-type adamw8bit --scheduler-type cosine_restarts \
  --warmup-steps 100 --weight-decay 0.01 \
  --max-grad-norm 1.0 --seed 42 \
  --num-workers 4 --log-every 10 \
  --device "{DEVICE}" --precision bf16 {main_resume_option}
if _exit_code != 0:
    raise RuntimeError(f'본학습이 종료 코드 {_exit_code}로 실패했습니다.')

FINAL_ADAPTER_DIR = LOCAL_RUN_DIR / 'final'
assert (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file(), 'LoRA 본학습 결과가 없습니다.'
shutil.copytree(LOCAL_RUN_DIR, LORA_OUTPUT_DIR, dirs_exist_ok=True)
print('full training saved to Drive:', LORA_OUTPUT_DIR)

In [ ]:
checkpoints = sorted(LORA_OUTPUT_DIR.rglob('*'))
print('output files:', len(checkpoints))
for path in checkpoints[-30:]:
    if path.is_file():
        print(path.relative_to(LORA_OUTPUT_DIR), path.stat().st_size)

## 6. 학습된 dual-stream LoRA inference

현재 `LORA_VERSION`에서 validation loss가 가장 낮은 checkpoint를 선택합니다. `/content/motif.wav`, 악기 이름, 언어, 가사와 생성 길이를 직접 입력해 accompaniment/vocal stem을 공동 생성하며 `CFG_SCALE`로 두 prompt를 강화합니다.

In [ ]:
from pathlib import Path
import gc
import json
import re
import soundfile as sf
import torch
from peft import PeftModel

from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.dual_stream import attach_dual_stream_conditioner
from acestep.training_v2.dual_stream_inference import sample_dual_stream_latents
from acestep.training_v2.dual_stream_preprocess import encode_motif_condition_latents
from acestep.training_v2.model_loader import (
    load_decoder_for_training, load_text_encoder, load_vae, unload_models,
)
from acestep.training_v2.preprocess_prompt import build_simple_prompt

# 학습 셀을 실행하지 않아도 Drive checkpoint만으로 inference할 수 있습니다.
LORA_VERSION = 'v6_repeated_motif'
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/MOHIM/mohim_lora_checkpoints')
DRIVE_CHECKPOINT_DIR = DRIVE_CHECKPOINT_ROOT / LORA_VERSION
INFERENCE_OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/inference') / LORA_VERSION
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
MODEL_VARIANT = 'base'
DEVICE = 'cuda'

def checkpoint_epoch(path):
    try:
        state = torch.load(
            path / 'training_state.pt', map_location='cpu', weights_only=False
        )
        return int(state.get('epoch', -1))
    except Exception:
        return -1

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_MOTIF_STEM = 'piano'
CUSTOM_MOTIF_START_SECONDS = 0.0  # 생성곡에서 이 시각을 원본 motif 시작점으로 고정
CUSTOM_LANGUAGE = 'English'
CUSTOM_LYRICS = '''
Paste the complete lyrics here.
'''
CUSTOM_OUTPUT_NAME = 'custom_song'
INFERENCE_DURATION_SECONDS = 60.0  # 생성할 곡 길이
INFERENCE_STEPS = 50
INFERENCE_SEED = 42
CFG_SCALE = 7.0
checkpoint_pattern = re.compile(r'^epoch_(\d+)_loss_([0-9.]+)$')
checkpoint_candidates = []
checkpoint_search_dirs = [DRIVE_CHECKPOINT_DIR]
if LORA_VERSION == 'v1':
    checkpoint_search_dirs.append(DRIVE_CHECKPOINT_ROOT)
for checkpoint_dir in checkpoint_search_dirs:
    for checkpoint_path in checkpoint_dir.glob('epoch_*_loss_*'):
        match = checkpoint_pattern.match(checkpoint_path.name)
        if (
            match
            and (checkpoint_path / 'adapter_model.safetensors').is_file()
            and (checkpoint_path / 'dual_stream_conditioner.pt').is_file()
        ):
            checkpoint_candidates.append((float(match.group(2)), int(match.group(1)), checkpoint_path))
    for checkpoint_path in (checkpoint_dir / 'resume_latest', checkpoint_dir / 'resume_latest.previous'):
        config_path = checkpoint_path / 'run_config.json'
        if (
            (checkpoint_path / 'adapter_model.safetensors').is_file()
            and (checkpoint_path / 'dual_stream_conditioner.pt').is_file()
            and config_path.is_file()
        ):
            run_config = json.loads(config_path.read_text(encoding='utf-8'))
            validation_loss = run_config.get('val_loss')
            epoch = checkpoint_epoch(checkpoint_path)
            if validation_loss is not None and epoch >= 0:
                checkpoint_candidates.append((float(validation_loss), epoch, checkpoint_path))
assert checkpoint_candidates, f'{LORA_VERSION} checkpoint가 없습니다: {DRIVE_CHECKPOINT_DIR}'
BEST_LOSS, BEST_EPOCH, BEST_ADAPTER_DIR = min(checkpoint_candidates, key=lambda item: item[0])

assert CUSTOM_MOTIF_PATH.is_file(), f'motif 파일이 없습니다: {CUSTOM_MOTIF_PATH}'
assert CUSTOM_MOTIF_STEM.strip(), 'CUSTOM_MOTIF_STEM을 입력하세요.'
assert CUSTOM_LYRICS.strip(), 'CUSTOM_LYRICS를 입력하세요.'
assert CFG_SCALE >= 1.0, 'CFG_SCALE은 1 이상이어야 합니다.'

dtype = torch.bfloat16
motif_stem = CUSTOM_MOTIF_STEM.strip().lower()
motif_duration = sf.info(str(CUSTOM_MOTIF_PATH)).duration
generation_duration = INFERENCE_DURATION_SECONDS
accompaniment_caption = (
    f'Full instrumental accompaniment for a pop song, organized around a recurring {motif_stem} motif. '
    'Complete rhythm, harmony, and instrumentation; no vocals.'
)
vocal_caption = (
    f'Isolated vocal stem for an {CUSTOM_LANGUAGE.strip()} pop song, singing the provided lyrics. '
    'Vocals only; no instrumental accompaniment and no musical instruments.'
)

print(f'version: {LORA_VERSION}, best epoch: {BEST_EPOCH}, validation loss: {BEST_LOSS:.4f}')
print('adapter:', BEST_ADAPTER_DIR)
print('custom motif:', CUSTOM_MOTIF_PATH)
print('accompaniment caption:', accompaniment_caption)
print('vocal caption:', vocal_caption)
print('lyrics characters:', len(CUSTOM_LYRICS.strip()))
print('generation duration:', generation_duration, 'seconds')
print('CFG scale:', CFG_SCALE)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
target_samples = round(generation_duration * 48000)
custom_motif_latents = encode_motif_condition_latents(
    str(CUSTOM_MOTIF_PATH),
    vae,
    dtype,
    target_samples=target_samples,
    motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + motif_duration,
)
output_frames = custom_motif_latents.shape[0]
custom_motif_mask = torch.ones(
    custom_motif_latents.shape[0], dtype=dtype
)
unload_models(vae)
del vae

text_tokenizer, text_encoder = load_text_encoder(
    CHECKPOINT_DIR, device=DEVICE, precision='bf16'
)
def encode_manual_prompt(caption, lyrics):
    prompt = build_simple_prompt(
        {
            'caption': caption, 'duration': generation_duration,
            'bpm': None, 'timesignature': '', 'keyscale': '',
        }
    )
    text_hs, text_mask = encode_text(
        text_encoder, text_tokenizer, prompt, DEVICE, dtype
    )
    lyric_hs, lyric_mask = encode_lyrics(
        text_encoder, text_tokenizer, lyrics, DEVICE, dtype
    )
    return text_hs, text_mask, lyric_hs, lyric_mask

accompaniment_text = encode_manual_prompt(accompaniment_caption, '[Instrumental]')
vocal_text = encode_manual_prompt(vocal_caption, CUSTOM_LYRICS.strip())
unload_models(text_encoder)
del text_encoder, text_tokenizer

model = load_decoder_for_training(
    CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16'
)
accompaniment_encoder_hs, accompaniment_encoder_mask = run_encoder(
    model, accompaniment_text[0], accompaniment_text[1], accompaniment_text[2], accompaniment_text[3], DEVICE, dtype
)
vocal_encoder_hs, vocal_encoder_mask = run_encoder(
    model, vocal_text[0], vocal_text[1], vocal_text[2], vocal_text[3], DEVICE, dtype
)
del accompaniment_text, vocal_text
model.decoder = PeftModel.from_pretrained(
    model.decoder, str(BEST_ADAPTER_DIR), is_trainable=False
).to(device=DEVICE, dtype=dtype).eval()

conditioner_config = json.loads(
    (BEST_ADAPTER_DIR / 'dual_stream_config.json').read_text(encoding='utf-8')
)
assert conditioner_config.pop('schema_version') == 2, '구형 dual-stream checkpoint입니다.'
conditioner = attach_dual_stream_conditioner(
    model.decoder, **conditioner_config
).to(DEVICE, dtype=dtype).eval()
conditioner.load_state_dict(
    torch.load(
        BEST_ADAPTER_DIR / 'dual_stream_conditioner.pt',
        map_location=DEVICE,
        weights_only=True,
    )
)

with torch.inference_mode():
    generated_accompaniment_latents, generated_vocal_latents = sample_dual_stream_latents(
        decoder=model.decoder,
        conditioner=conditioner,
        accompaniment_encoder_hidden_states=accompaniment_encoder_hs.to(DEVICE, dtype=dtype),
        accompaniment_encoder_attention_mask=accompaniment_encoder_mask.to(DEVICE, dtype=dtype),
        vocal_encoder_hidden_states=vocal_encoder_hs.to(DEVICE, dtype=dtype),
        vocal_encoder_attention_mask=vocal_encoder_mask.to(DEVICE, dtype=dtype),
        motif_seed_latents=custom_motif_latents.unsqueeze(0).to(DEVICE, dtype=dtype),
        motif_seed_attention_mask=custom_motif_mask.unsqueeze(0).to(DEVICE),
        null_condition_emb=model.null_condition_emb.to(DEVICE, dtype=dtype),
        output_frames=output_frames,
        latent_dim=custom_motif_latents.shape[-1],
        steps=INFERENCE_STEPS,
        seed=INFERENCE_SEED,
        guidance_scale=CFG_SCALE,
    )
generated_accompaniment_latents = generated_accompaniment_latents.cpu()
generated_vocal_latents = generated_vocal_latents.cpu()
del model, conditioner, custom_motif_latents
gc.collect()
torch.cuda.empty_cache()
print('custom motif-conditioned accompaniment/vocal generation complete')

In [ ]:
import math
import torch.nn.functional as F
import soundfile as sf
from IPython.display import Audio, display
from acestep.training_v2.model_loader import load_vae

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    latent_frames = latents.shape[-1]
    stride = chunk_frames - 2 * overlap
    if stride <= 0:
        raise ValueError('chunk_frames must be larger than twice overlap')
    decoded = []
    steps = math.ceil(latent_frames / stride)
    upsample_factor = None
    for index in range(steps):
        core_start = index * stride
        core_end = min(core_start + stride, latent_frames)
        window_start = max(0, core_start - overlap)
        window_end = min(latent_frames, core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        if upsample_factor is None:
            upsample_factor = audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * upsample_factor)
        trim_end = round((window_end - core_end) * upsample_factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
        del chunk, audio
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
accompaniment_audio = decode_latents_tiled(vae, generated_accompaniment_latents)
vocal_audio = decode_latents_tiled(vae, generated_vocal_latents)
del vae, generated_accompaniment_latents, generated_vocal_latents
gc.collect()
torch.cuda.empty_cache()

output_sample_rate = 48000
target_samples = round(generation_duration * output_sample_rate)
def match_length(audio):
    if audio.shape[-1] < target_samples:
        audio = F.pad(audio, (0, target_samples - audio.shape[-1]))
    return audio[:, :, :target_samples]
accompaniment_audio = match_length(accompaniment_audio)
vocal_audio = match_length(vocal_audio)
preview_mix = accompaniment_audio + vocal_audio
preview_mix = preview_mix / preview_mix.abs().amax().clamp_min(1.0)

track_output_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME
track_output_dir.mkdir(parents=True, exist_ok=True)
outputs = {
    'accompaniment': track_output_dir / 'generated_accompaniment.wav',
    'vocals': track_output_dir / 'generated_vocals.wav',
    'preview_mix': track_output_dir / 'generated_preview_mix.wav',
}
for name, path in outputs.items():
    audio = {'accompaniment': accompaniment_audio, 'vocals': vocal_audio, 'preview_mix': preview_mix}[name]
    sf.write(path, audio.squeeze(0).transpose(0, 1).numpy(), output_sample_rate)
    print(name, path, sf.info(path).duration, 'seconds')
    display(Audio(filename=str(path)))